Name:   Srujana Duggineni

ID:     C00313483

#### Algorithm: Random Forest,Decision tree (Hyperparameter Tuning,)
Dataset: [ Germany Used Cars Dataset 2023 ](https://www.kaggle.com/datasets/wspirat/germany-used-cars-dataset-2023)

Business Understanding: 

Objective:
The target of this work is to perform Random Forest algorithm and compare its working performance with Hyperparameter Tuning,Feature Importance  . Such comparison also enables the user to gain insights into the internal working of ensemble methods, especially Random Forest.

##### Data Understanding  
##### Features:
Brand: The brand or manufacturer of the car.

Model: The specific model of the car.

Color: The color of the car's exterior.

Registration Date: The date when the car was registered (Month/Year).

Year of Production: The year in which the car was manufactured.

Price in Euro: The price of the car in Euros.

Power: The power of the car in kilowatts (kW) and horsepower (ps).

Transmission Type: The type of transmission (e.g., automatic, manual).

Fuel Type: The type of fuel the car requires.

Fuel Consumption: Information about the car's fuel consumption in L/100km ang g/km.

Mileage: The total distance traveled by the car in km.

Offer Description: Additional description provided in the car offer.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from preprocess_data import preprocess_data
from sklearn.utils import resample

#### Preparing dataset , partition data

The following EDA steps are followed (inside preprocess_data):
    1. Load dataset.
    2. Drop unwanted columns.
    3. Handle missing values.
    4. Encode categorical features.
    5. Process datetime features(NA)
    6. Remove outliers.
    7. Scale numerical features.
    8. Split into train & test sets.
    

In [2]:
# preprocessing dataset for classification
file_path = "/Users/dugginenisrujana/Desktop/DAA_CA /DAA_CA1/Datasets/GermanyUsedCarsDataset2023.csv"

### the column we want to predict
target_column = "price_in_euro" 
drop_columns = ["Unnamed: 0"]

X_train, X_test, y_train, y_test = preprocess_data(file_path, target_column, drop_columns=drop_columns)

1. Load dataset.

First 5 Rows:
    Unnamed: 0          brand              model  color registration_date  \
0       93699    lamborghini        Lamborghini   grey            18-Aug   
1       93840    lamborghini        Lamborghini  black            21-Nov   
2      106583  mercedes-benz  Mercedes-Benz SLR   grey             7-Jan   
3        1509   aston-martin       Aston Martin  brown            11-Aug   
4      165445        porsche        Porsche 918  white            14-Jun   

   year  price_in_euro  power_kw  power_ps transmission_type fuel_type  \
0  2018        5890500     566.0     770.0         Automatic    Petrol   
1  2021        3250000     602.0     818.0         Automatic    Hybrid   
2  2007        2490000     478.0     650.0         Automatic    Petrol   
3  2011        2289000     559.0     760.0         Automatic    Petrol   
4  2014        1990000     652.0     886.0         Automatic    Hybrid   

  fuel_consumption_l_100km fuel_consumption_g_km  mileage_in_km  

#### Modelling  

Decision Tree focuses on splitting data based on Gini impurity to make predictions, while a Random Forest combines multiple decision trees for improved accuracy and robustness through majority voting.

#### Model training and evalution

In [3]:
#### Handling class imbalance
from imblearn.over_sampling import SMOTE


# Convert pandas DataFrames/Series to numpy arrays for custom model
X_train_np = X_train.to_numpy() if isinstance(X_train, pd.DataFrame) else X_train
y_train_np = y_train.to_numpy() if isinstance(y_train, pd.Series) else y_train
X_test = np.array(X_test)


# Convert target values to integers (for classification)
y_train_np = y_train_np.astype(int)
y_test = y_test.astype(int)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_np, y_train_np)
# # Remove rows with negative values in the target variable
mask = y_train_balanced >= 0
X_train_clean = X_train_balanced[mask]
y_train_clean = y_train_balanced[mask]

# Train and evaluate library Random Forest model (scikit-learn)
rf_lib = RandomForestClassifier(n_estimators=100,  # Number of trees (adjust as needed)
                                      max_depth=None,       # Maximum depth of trees 
                                      min_samples_split=2,  # Minimum samples for split
                                      min_samples_leaf=1,   # Minimum samples in leaf
                                      random_state=42,      # For reproducibility
                                      n_jobs=-1)
rf_lib.fit(X_train_balanced, y_train_balanced)
y_pred_lib = rf_lib.predict(X_test)
accuracy_lib = accuracy_score(y_test,y_pred_lib)

# Print the evaluation results

print(f"Random Forest (Library) Accuracy: {accuracy_lib:.4f}")


Random Forest (Library) Accuracy: 0.9090


In [4]:
# 5. Evaluate the model
from sklearn.metrics import accuracy_score, classification_report
accuracy = accuracy_score(y_test, y_pred_lib)
print(f"Random Forest (scikit-learn) Accuracy: {accuracy:.4f}")

# Classification report for more detailed metrics
print(classification_report(y_test, y_pred_lib))

Random Forest (scikit-learn) Accuracy: 0.9090
              precision    recall  f1-score   support

           0       0.97      0.96      0.97      1344
           1       0.58      0.64      0.61       157
           2       0.71      0.68      0.69       121
           3       0.00      0.00      0.00         4

    accuracy                           0.91      1626
   macro avg       0.57      0.57      0.57      1626
weighted avg       0.91      0.91      0.91      1626



Random Forest (scikit-learn) Accuracy: 0.9090


## Additional 

In [5]:
# Feature Importance (Optional)
feature_importances = rf_lib.feature_importances_
print("Feature Importances:", feature_importances)
feature_importance_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': feature_importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
print(feature_importance_df)

Feature Importances: [0.08856456 0.13127845 0.03568601 0.07302139 0.07576811 0.14558021
 0.15877116 0.         0.01140783 0.06145941 0.05010577 0.07644354
 0.09191356]
                     Feature  Importance
6                   power_ps    0.158771
5                   power_kw    0.145580
1                      model    0.131278
12         offer_description    0.091914
0                      brand    0.088565
11             mileage_in_km    0.076444
4                       year    0.075768
3          registration_date    0.073021
9   fuel_consumption_l_100km    0.061459
10     fuel_consumption_g_km    0.050106
2                      color    0.035686
8                  fuel_type    0.011408
7          transmission_type    0.000000


Observation:
    
    power_ps    0.158824;==>top most important feature 

    power_kw    0.145983;==>2nd most important feature 

     model    0.130769;==>3rd most important feature 

#### Hyperparameter Tuning

In [6]:
# 6. Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 75, 100],      
    'max_depth': [None, 2, 5, 5],       
    'min_samples_split': [2, 5, 5],   
    'min_samples_leaf': [1, 2, 3]       
}

grid_search = GridSearchCV(estimator=rf_lib, param_grid=param_grid, cv=3, n_jobs=-1, verbose=1, scoring='accuracy')  # Use cross-validation
grid_search.fit(X_train_balanced, y_train_balanced)

best_rf = grid_search.best_estimator_
print("Best Hyperparameters:", grid_search.best_params_)

y_pred_best = best_rf.predict(X_test)
accuracy_best = accuracy_score(y_test, y_pred_best)
print(f"Random Forest (scikit-learn - Tuned) Accuracy: {accuracy_best:.4f}")
print(classification_report(y_test, y_pred_best))

Fitting 3 folds for each of 108 candidates, totalling 324 fits
Best Hyperparameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Random Forest (scikit-learn - Tuned) Accuracy: 0.9090
              precision    recall  f1-score   support

           0       0.97      0.96      0.97      1344
           1       0.58      0.64      0.61       157
           2       0.71      0.68      0.69       121
           3       0.00      0.00      0.00         4

    accuracy                           0.91      1626
   macro avg       0.57      0.57      0.57      1626
weighted avg       0.91      0.91      0.91      1626



#### Saving model(future use)

In [7]:
import joblib  # Import joblib to save the model

# Saving the Custom Model
joblib.dump(best_rf, "best_RFdt_model.pkl")


print("Models saved successfully!")

Models saved successfully!
